# Rare-event simulation: running an IPS estimate

A loss of separation, under a competent separation manager, is *rare* — here around one in ten thousand encounters or rarer. Plain Monte Carlo has to run on the order of $1/P$ encounters before it sees one, and a batch that reads **zero** events tells you nothing. The rare-event estimator in [`opencdarr.ips`](https://github.com/fazlurnu/OpenCDaRR/blob/main/opencdarr/ips.py) — a Blom–Bakker **interacting particle system** (importance splitting) — instead steers a fixed population of particles toward the rare set through a ladder of shrinking separation shells, and returns the small probability with a confidence interval.

This notebook runs a **quick trial** so you can see the moving parts. The theory (why it is unbiased, why the *running* minimum) and the full validation live in the handbook's rare-event pages; production runs use far more particles (see the note at the end).

In [ ]:
# Install once, from the repo root:  pip install -e ".[examples]"
# (the examples extra pulls in opencdarr[parallel], i.e. joblib, for opencdarr.parallel)
# %pip install -e "..[examples]"

## The scenario

One fixed encounter — two multirotors on a 90° collision course (zero design miss), each carrying a GNSS self-fix error. The separation manager (`StateBased` + `MVP` + `PastCPA`) clears them to just outside the 50 m protected zone almost every time; loss of separation is the thin tail where the navigation noise makes an aircraft under-clear.

IPS needs two things: a **factory** that builds one particle (its rules + initial world) from a seed, and the **shell ladder** — a decreasing sequence of running-minimum separations ending at `rpz`. Nothing else is scenario-specific.

In [ ]:
import time

from opencdarr.ips import Particle
from opencdarr.parallel import estimate_rare_prob
from opencdarr.fleet import Agent, build_env
from opencdarr.performance import M600
from opencdarr.scenario import create_conflict
from opencdarr.state import AircraftState
from opencdarr.cd import StateBased
from opencdarr.cr import MVP
from opencdarr.crr import PastCPA
from opencdarr.cns.navigation import GnssNavigation

def make_start():
    """The one starting particle: a fixed 90 deg crossing with GNSS noise, plus its rules."""
    own = AircraftState(id='OWN', lat=52.0, lon=4.0, trk=0.0, gs=10.2889,
                        pos_ci95=3.0, vel_ci95=1.0)          # 3 m / 1 m/s self-fix error
    intr = create_conflict(own, intr_id='INT', dpsi=90.0, dcpa=0.0,
                           tlos=70.0, rpz=50.0, side=1)       # 90 deg, dead-on, LoS 70 s out
    agents = [Agent(own, M600), Agent(intr, M600)]
    env = build_env(agents, rpz=50.0, t_lookahead=60.0, dt=0.5,
                    detector=StateBased(), resolver=MVP(margin=1.05),
                    recovery=PastCPA(bouncing_guard=True), navigation=GnssNavigation(),
                    done_timeout=10.0)
    return Particle(env=env, state=env.initial_state(agents))

# The geometry is pinned, so every particle starts from the same world under the same rules.
# Build it ONCE and share it: FleetEnv and FleetState are deeply immutable, so sharing is safe --
# and it keeps the particle cloud cheap to hand to worker processes.
START = make_start()

def build_initial(seq):
    """One IPS particle. Geometry is fixed here, so the seed feeds the forward noise, not this."""
    return START

# a decreasing ladder of running-min shells, ending at rpz = 50 m
LEVELS = [150, 135, 122, 112, 104, 97, 90, 82, 74, 68, 63, 59, 56, 54, 52, 51, 50]

## Run it — across replications *and* across particles

One IPS run (`ips_once`) evolves the $N$ particles shell by shell — survivors are resampled back to $N$ at each shell — and returns $\hat P = \prod_k (S_k/N)$. Within a single run the particles interact (they share ancestors), so a single run's spread is **not** a valid confidence interval. The honest CI comes from **independent replications** on separate seed subtrees.

Those replications parallelise perfectly, one per core — but that caps you at `REPS` cores, and `REPS` is a *statistical* choice, not a hardware one. So there is a second axis. Within one shell, every particle evolves on its own pre-assigned RNG substream and reads nothing from its neighbours, so the particles can be split across workers too; the only synchronisation point is the resampling barrier at the end of each shell. [`opencdarr.parallel`](https://github.com/fazlurnu/OpenCDaRR/blob/main/opencdarr/parallel.py) uses whichever axis fits the machine, and because the substreams are addressed by index rather than handed out in sequence, the answer is **identical to the last bit** however the work was divided. Pick `REPS` for the confidence interval you want and `n_jobs` for the machine you have.

In [ ]:
N_PARTICLES = 2000    # per shell (production uses ~10000; see the note below)
REPS = 8              # independent replications -> the confidence interval

t0 = time.perf_counter()
est = estimate_rare_prob(build_initial, LEVELS, n_particles=N_PARTICLES,
                         reps=REPS, seed=20260728, n_jobs=-1)
elapsed = time.perf_counter() - t0

print(f'P(LoS) = {est.prob:.2e}   95% CI [{est.ci[0]:.2e}, {est.ci[1]:.2e}]')
print(f'collapsed replications: {est.n_collapsed}/{REPS}   ({elapsed:.0f} s)')

The estimate is reported as **`P ± CI`, never a bare probability**. The CI, from `REPS` independent runs, is combined in log space because the product estimator is right-skewed.

**Watch `collapsed`.** A replication *collapses* when some shell ends with zero survivors — it then returns $\hat P = 0$, which biases the average **low**. A collapse means the ladder is too aggressive or the particle count too small for this rarity; it is a failed run, not a data point. Here it should read `0/8`.

The per-shell survival fraction is the diagnostic behind the estimate — each should stay healthy (roughly 0.2–0.8). A shell that survives near 0 is about to collapse; one near 1.0 is wasted.

In [4]:
good = [r for r in est.reps if r.collapsed_at is None]
levels = good[0].levels
mean_surv = [sum(r.survival[k] for r in good) / len(good) for k in range(len(levels))]
print('shell (m):  survival')
for d, s in zip(levels, mean_surv):
    bar = '#' * round(s * 40)
    print(f'  {d:5.0f}  :  {s:.2f}  {bar}')

shell (m):  survival
    150  :  0.49  ####################
    135  :  0.85  ##################################
    122  :  0.85  ##################################
    112  :  0.87  ###################################
    104  :  0.87  ###################################
     97  :  0.87  ###################################
     90  :  0.85  ##################################
     82  :  0.79  ###############################
     74  :  0.73  #############################
     68  :  0.72  #############################
     63  :  0.64  ##########################
     59  :  0.56  ######################
     56  :  0.42  #################
     54  :  0.40  ################
     52  :  0.19  ########
     51  :  0.31  ############
     50  :  0.14  ######


## From a trial to a production estimate

This trial keeps things quick — `dt = 0.5 s`, 2000 particles, 8 replications — so the CI is loose and the point estimate carries some discretisation bias. The CNS-uncertainty sweep behind the handbook's validation page runs the same machinery at production settings — **`dt = 0.2 s`, 10000 particles, 10 replications, over a 2-million-encounter Monte-Carlo anchor per cell** — and confirms IPS agrees with Monte Carlo across the navigation/communication space, at probabilities from $10^{-4}$ down toward $10^{-5}$ where Monte Carlo starves.

To scale this trial up: raise `N_PARTICLES` (more headroom against collapse), raise `REPS` (tighter CI), and set `dt=0.2`. Everything else stays exactly as above. Note that raising `REPS` no longer has anything to do with filling the machine — `n_jobs=-1` does that on its own, whatever `REPS` is — so choose it purely for the interval width you need, remembering it costs compute in proportion.